# Generate user_kyc Data
Update DB credentials then Run All.

In [ ]:
import random
from datetime import timedelta
import pandas as pd
import psycopg2
from psycopg2.extras import execute_batch

DB={
 'host':'localhost',
 'port':5432,
 'database':'credict',
 'user':'postgres',
 'password':'YOUR_PASSWORD'
}

conn=psycopg2.connect(**DB)
cur=conn.cursor()

users=pd.read_sql("SELECT user_id, signup_date, acquisition_channel, city_tier FROM users ORDER BY user_id;",conn)

providers=['Signzy','HyperVerge','Karza','Surepass']
reasons=['PAN mismatch','Aadhaar failed','Blurred selfie','Duplicate account','Fraud suspected','Manual review failed']

channel_factor={'Referral':0.98,'Organic':0.95,'Paid':0.90}
tier_factor={'Tier 1':1.00,'Tier 2':0.98,'Tier 3':0.96}

rows=[]
for _,u in users.iterrows():
    signup=u.signup_date
    ch=channel_factor.get(u.acquisition_channel,0.95)
    tf=tier_factor.get(u.city_tier,0.98)

    p_phone=min(0.995,ch)
    p_pan=min(0.99,ch)
    p_panv=0.99
    p_aad=0.97*tf
    p_self=0.96*tf
    p_app=0.95*ch

    phone=pan=panv=aad=selfie=False
    phone_at=pan_at=panv_at=aad_at=selfie_at=app_at=None
    status='pending';rej=None

    if random.random()<p_phone:
        phone=True; phone_at=signup+timedelta(minutes=random.randint(1,30))
        if random.random()<p_pan:
            pan=True; pan_at=phone_at+timedelta(minutes=random.randint(5,180))
            if random.random()<p_panv:
                panv=True; panv_at=pan_at+timedelta(minutes=random.randint(2,60))
                if random.random()<p_aad:
                    aad=True; aad_at=panv_at+timedelta(minutes=random.randint(5,120))
                    if random.random()<p_self:
                        selfie=True; selfie_at=aad_at+timedelta(minutes=random.randint(1,45))
                        d=random.random()
                        if d<p_app:
                            status='approved'
                            app_at=selfie_at+timedelta(minutes=random.randint(2,20))
                        elif d<p_app+0.03:
                            status='rejected'
                            rej=random.choice(reasons)
                        else:
                            status='pending'
    rows.append((int(u.user_id),phone,phone_at,pan,pan_at,panv,panv_at,aad,aad_at,selfie,selfie_at,status,app_at,rej,random.choice(providers),signup,signup))

sql="""INSERT INTO user_kyc
(user_id,phone_verified,phone_verified_at,pan_submitted,pan_submitted_at,
pan_verified,pan_verified_at,aadhaar_linked,aadhaar_linked_at,
selfie_submitted,selfie_submitted_at,kyc_status,kyc_approved_at,
rejection_reason,kyc_provider,created_at,updated_at)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)"""

execute_batch(cur,sql,rows,page_size=5000)
conn.commit()

print('Inserted',len(rows),'records')

summary=pd.read_sql("""
SELECT COUNT(*) total,
COUNT(kyc_approved_at) approved,
ROUND(COUNT(kyc_approved_at)*100.0/COUNT(*),2) approval_rate
FROM user_kyc;
""",conn)
print(summary)

cur.close(); conn.close()
